In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)

x = torch.randn(1, 3, 32, 32)  # 每批次一张图片，3 个通道，大小是 32x32
y = conv(x)

print(y.shape) # 一张图片经过卷积层后，变成了 16 张特征图，每张特征图的大小是 30x30
plt.imshow(y[0, 0].detach().numpy(), cmap="gray") # 显示第一张特征图
plt.title("Feature Map 1")
plt.axis("off")

In [ ]:
import torch
import matplotlib.pyplot as plt

# 1. 生成假图：(批次, 通道, 高, 宽) = (1, 3, 32, 32)
x = torch.randn(1, 3, 32, 32)

# 2. 把张量转成可以画的格式
# 先把批次去掉，变成 (3, 32, 32)
img_tensor = x[0]

# 把 PyTorch 的 [C, H, W] 转成 Matplotlib 需要的 [H, W, C]
img_np = img_tensor.permute(1, 2, 0).numpy()

# 3. 把随机值缩到 0-1 之间，不然颜色会很奇怪
img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

# 4. 画出来
plt.figure(figsize=(4, 4))
plt.imshow(img_np)
plt.axis('off')  # 关掉坐标轴
plt.title("Random Fake Image")
plt.show()

In [ ]:
# 导入 PyTorch 核心库，用来搭建神经网络
import torch

# 导入神经网络层模块，这里用来定义卷积层
import torch.nn as nn

# 导入画图工具库，用来显示图片和卷积提取的特征图
import matplotlib.pyplot as plt

# 导入 torchvision，专门用来处理图像、加载数据集
import torchvision

# 导入图像预处理工具，比如把图片转成张量
import torchvision.transforms as transforms

# ===================== 1. 数据准备 =====================
# 定义图像预处理操作：只做一件事 -> 把图片转成 PyTorch 张量（Tensor）
# 张量就是模型能看懂的数字格式
transform = transforms.Compose([
    transforms.ToTensor()
])

# 加载 CIFAR-10 测试数据集（1万张图片）
# root='./data'：数据集存放在当前目录的 data 文件夹下
# train=False：加载**测试集**（试卷），不是训练集（课本）
# download=True：如果本地没有数据集，自动下载；有就直接加载
# transform=transform：加载图片时，自动执行上面定义的预处理（转张量）
dataset = torchvision.datasets.CIFAR10(
    root='.././data',
    train=False,
    download=True,
    transform=transform
)

# 从测试集中取出**第 0 张**图片和它对应的标签（类别）
img, label = dataset[1]

# 给图片增加一个 batch 维度，把形状从 [3, 32, 32] 变成 [1, 3, 32, 32]
# 因为神经网络模型默认接受一批数据输入，单张图片也要加个维度
x = img.unsqueeze(0)

# ===================== 2. 搭建卷积层 =====================
# 创建一个卷积层 Conv2d
# 3：输入通道数（彩色图片 R、G、B 3个通道）
# 8：输出通道数（卷积后生成 8 张特征图）
# kernel_size=3：卷积核大小 3x3
# padding=1：给图片边缘填充 1 圈像素，保证卷积后图片大小不变
conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)

# ===================== 3. 卷积运算（前向传播） =====================
# 把图片 x 输入到卷积层，得到输出 y
# y 就是卷积提取后的**特征图**，形状是 [1, 8, 32, 32]
y = conv(x)

# ===================== 4. 显示原始图片 =====================
# img.permute(1, 2, 0)：把张量形状从 [3, 32, 32] 转成 [32, 32, 3]
# 因为画图工具要求 宽×高×通道 格式
plt.imshow(img.permute(1, 2, 0))

# 设置图片标题
plt.title("Original Image")

# 关闭坐标轴，让图片更好看
plt.axis('off')

# 弹出窗口显示原图
plt.show()

# ===================== 5. 显示卷积提取的特征图 =====================
# 创建一个 1行6列 的子图窗口，用来显示 6 张特征图
fig, axes = plt.subplots(1, 8, figsize=(12, 4))

# 循环显示前 8 张特征图
for i in range(8):
    # y[0, i]：取第 0 个batch 的第 i 张特征图
    # detach().numpy()：把张量转成普通数字数组，才能画图
    # cmap='gray'：用灰度图显示（黑白）
    axes[i].imshow(y[0, i].detach().numpy(), cmap='gray')
    
    # 关闭子图坐标轴
    axes[i].axis('off')

# 弹出窗口显示所有特征图
plt.show()

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO

# 👉 换成国内超快、绝对能打开的猫咪图片！
url = "https://picsum.photos/id/40/500/300"

# 读取图片
response = requests.get(url)
img = Image.open(BytesIO(response.content)).convert("RGB")
print(f"Original image size: {img.size}")
plt.figure()
plt.imshow(img)
plt.show()   # 出图1

# 预处理
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # 改成和 CIFAR 一样大小
    transforms.ToTensor(),
])

img_tensor: torch.Tensor = transform(img) # type: ignore
x = img_tensor.unsqueeze(0)
print(f"Preprocessed image shape: {x.shape}")  # 出图2

# 卷积层
conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)

# 卷积运算
y = conv(x)

# 显示原图
plt.imshow(img_tensor.permute(1, 2, 0))
plt.title("Original Image")
plt.axis("off")
plt.show()

# 显示卷积特征
fig, axes = plt.subplots(1, 8, figsize=(12, 4))
for i in range(8):
    axes[i].imshow(y[0, i].detach().numpy(), cmap="gray")
    axes[i].axis("off")
plt.show()